In [1]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
from openai import OpenAI

BASE_URL = "http://gpustack.cds.utsa.edu/v1"
API_KEY = "REPLACE-WITH-YOUR-KEY"

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

models = client.models.list().data

for model in models:
    print(model.id)

MODEL = models[0].id
print("Connected. Model:", MODEL)

qwen3.5-122b-a10b
qwen3-vl-8b-instruct
qwen3-coder-next
Connected. Model: qwen3.5-122b-a10b


In [6]:
import time

def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""
    
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                top_p=0.95,
            )
            return resp.choices[0].message.content

        except Exception:
            if attempt == retries:
                raise

            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))



Hello!


In [8]:
question = (
    "I am writing a Python function triage_log(line) that asks an LLM to classify a "
    "security log line and returns a dataclass with label and reason. Suggest a robust "
    "prompt format and a defensive way to parse the reply. Keep it concise."
)

print(ask_model(question))



```python
import json
import re
from dataclasses import dataclass
from typing import Optional

@dataclass
class TriagedLog:
    label: str
    reason: str

ALLOWED_LABELS = {"benign", "suspicious", "malicious", "unknown"}

def triage_log(line: str, llm_client) -> TriagedLog:
    prompt = f"""
    Classify this security log line. Output strictly valid JSON.
    Labels: {', '.join(ALLOWED_LABELS)}
    Format: {{"label": "<label>", "reason": "<string>"}}
    Log: {line}
    """
    
    # 1. Call LLM (assumed to return raw string)
    response = llm_client.generate(prompt, temperature=0.0)
    
    # 2. Defensive JSON Extraction (handle markdown wrappers)
    json_match = re.search(r'\{.*\}', response, re.DOTALL)
    if not json_match:
        return TriagedLog(label="unknown", reason="parse_error")
        
    try:
        data = json.loads(json_match.group(0))
        label = data.get("label", "unknown").lower()
        reason = data.get("reason", "no_reason_provided")
        
     

In [17]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(
    r"^LABEL:\s*(\w+)",
    re.IGNORECASE | re.MULTILINE
)

REASON_RE = re.compile(
    r"^REASON:\s*(.+)",
    re.IGNORECASE | re.MULTILINE
)

In [18]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS and give a one "
        "sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )

    text = ask_model(prompt)

    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"

    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN

    reason_match = REASON_RE.search(text)
    reason = (
        reason_match.group(1).strip()
        if reason_match
        else text.strip()
    )

    return Triage(
        line=line,
        label=label,
        reason=reason
    )

In [ ]:
TOOL_NAME = "LandaScan"
TAGLINE = "Intelligent Threat Detection for Security Logs"

print("Tool Name:", TOOL_NAME)

In [22]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='This failed login attempt for the root user from an external IP is a common indicator of brute-force attacks.')


In [24]:
LOG_FILE = "AM SecLogs 11162026.csv"
MAX_ENTRIES = 20

def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines[:limit]

entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries]

print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from AM SecLogs 11162026.csv


In [28]:
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")

        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")

    return path

report_path = write_report(results, TOOL_NAME)

print("Wrote", report_path, "\n")
print(open(report_path, encoding="utf-8").read())

Wrote assessment.md 

# LandaScan Security Log Assessment

Generated: 2026-11-16 12:32

Entries assessed: 20

- **BENIGN** This is a metadata header row from a log export and does not represent an actionable security event.
  - `﻿Keywords,Date and Time,Source,Event ID,Task Category`
- **BENIGN** This is a standard Windows security audit event that frequently occurs during routine user logins and permission verification processes.
  - `Audit Success,11/16/2026 12:12:01 PM,Microsoft-Windows-Security-Auditing,4798,User Account Management,"A user's local group membership was enumerated.`
- **BENIGN** This log entry is an incomplete fragment containing no actionable security data or threat indicators.
  - `Subject:`
- **BENIGN** This indicates the built-in SYSTEM account used by the operating system for core processes, which is standard behavior in Windows security logs.
  - `Security ID:		SYSTEM`
- **BENIGN** The trailing dollar sign indicates a standard Windows machine account, which is n

Why model the output with an Enum and dataclass instead of a bare dict?
Enum helps us label raw data values as the set values, Benign, Suspicious, Maclicious or Unknown which keeps all the data simple. Dataclass supports this by structuting the object with named fields making easier to analyze quickly.
What does your defensive parsing do when the model ignores the format?
the defensive parsing uses the set label and reason to search into the labels mentioned above. If it does not match it will categorize it as unknown.
Which entries in your report look wrong, and how would you catch that?
If data is not able to fit in one of the four labels mentioned above it will be easier to spot as it will not have a label and raise concern for further inspection.